In [1]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

# ==============================
# 2. Load Dataset
# ==============================
df = pd.read_csv("OnlineRetail.csv", encoding="ISO-8859-1")

# ==============================
# 3. Data Preprocessing
# ==============================
# Remove missing customers
df = df[df['CustomerID'].notna()]

# Remove invalid prices
df = df[df['UnitPrice'] > 0]

# ==============================
# 4. Create User-Item Matrix
# ==============================
ratings = df.groupby(['CustomerID', 'StockCode'])['Quantity'].sum().reset_index()

user_item = ratings.pivot(index='CustomerID',
                          columns='StockCode',
                          values='Quantity').fillna(0)

print("User-Item Matrix Shape:", user_item.shape)

# ==============================
# 5. Apply Clustering (K-Means)
# ==============================
kmeans = KMeans(n_clusters=5, random_state=42)
user_clusters = kmeans.fit_predict(user_item)

# Add cluster labels to users
user_item['Cluster'] = user_clusters

print(user_item[['Cluster']].head())

# ==============================
# 6. Compute Item Similarity
# ==============================
# Remove cluster column for similarity
user_item_cf = user_item.drop('Cluster', axis=1)

# Transpose (items as rows)
item_user = user_item_cf.T

# Cosine similarity
item_similarity = cosine_similarity(item_user)

item_sim_df = pd.DataFrame(item_similarity,
                           index=item_user.index,
                           columns=item_user.index)

# ==============================
# 7. Recommendation Function
# ==============================
def recommend_products(customer_id, user_item, item_sim_df, top_n=5):
    
    # Get cluster of the user
    cluster = user_item.loc[customer_id, 'Cluster']
    
    # Get users from same cluster
    cluster_users = user_item[user_item['Cluster'] == cluster].index
    
    # Remove cluster column
    user_item_cf = user_item.drop('Cluster', axis=1)
    
    # Get items bought by user
    bought = user_item_cf.loc[customer_id]
    bought = bought[bought > 0].index
    
    scores = {}

    # Calculate scores
    for item in item_sim_df.columns:
        if item not in bought:
            score = 0
            for b in bought:
                score += item_sim_df.loc[item, b] * user_item_cf.loc[customer_id, b]
            scores[item] = score

    # Sort and return top N
    recommendations = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    return pd.DataFrame(recommendations, columns=['Product', 'Score'])

# ==============================
# 8. Test Recommendation
# ==============================
sample_customer = user_item.index[10]

print("Recommendations for Customer:", sample_customer)
print(recommend_products(sample_customer, user_item, item_sim_df, top_n=5))

User-Item Matrix Shape: (4371, 3684)
StockCode   Cluster
CustomerID         
12346.0           0
12347.0           0
12348.0           0
12349.0           0
12350.0           0
Recommendations for Customer: 12357.0
  Product       Score
0   23240  565.441502
1   21509  534.402556
2   20725  529.753696
3   23192  521.278668
4   23121  520.156642
